# Fourier Approximation: Signals and Images

The **Fourier approximation** of a function $f$ retains only the $2r+1$ lowest-frequency Fourier coefficients and discards the rest. For a 1D signal of length $n$:
$$
f_r = \mathcal{F}^{-1}\bigl(\hat{f} \cdot \mathbf{1}_{|k| \leq r}\bigr),
$$
where $\hat{f}_k = \tfrac{1}{n}\sum_j f_j e^{-2\pi i jk/n}$. This is the best $L^2$ approximation in the span of the first $2r+1$ Fourier atoms.

## Convergence rates

The approximation quality depends on the regularity of $f$:
- If $f \in C^s$ (s times continuously differentiable), then $\|f - f_r\|_2 = O(r^{-s})$.
- For analytic functions (e.g. smooth bumps): **exponential** convergence.
- For functions with jump discontinuities: slow $O(1/r)$ convergence and **Gibbs oscillations** near the jumps.

## 2D Fourier approximation

For an $n\times n$ image $f$, the 2D Fourier transform is
$$
\hat{f}(k, \ell) = \sum_{x,y} f(x,y)\, e^{-2\pi i(kx+\ell y)/n}.
$$
A **bandlimited** approximation retains only modes in a disk of radius $r$ in the frequency domain:
$$
f_r = \mathcal{F}_2^{-1}\bigl(\hat{f} \cdot \mathbf{1}_{\sqrt{k^2+\ell^2} \leq r}\bigr).
$$
As $r$ increases, the image sharpens and level lines become more detailed.

## Environment

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider

plt.rcParams['figure.dpi'] = 120

## 1D test signals

We build four 1D signals with different regularity properties to study their Fourier approximation behavior:
- **Smooth bump**: infinitely differentiable, fast Fourier convergence.
- **Ramp**: piecewise linear, $C^0$ but not $C^1$, medium convergence.
- **Box**: jump discontinuity (piecewise constant), slow convergence + Gibbs.
- **Sawtooth**: combination of jumps and linear parts.

In [ ]:
n = 1024
t = np.linspace(0, 1, n, endpoint=False)

signals = {
    'smooth bump':  np.exp(-((t - 0.5)**2) / (2 * 0.06**2)),
    'piecewise ramp': np.where(np.abs(t - 0.5) < 0.2, t - 0.5, 0.0),
    'box':          (np.abs(t - 0.5) < 0.2).astype(float),
    'sawtooth':     (t - np.floor(t + 0.5)),
}

fig, axes = plt.subplots(1, 4, figsize=(13, 3))
for ax, (name, sig) in zip(axes, signals.items()):
    ax.plot(t, sig, 'b-', lw=2)
    ax.set_title(name, fontsize=9); ax.grid(alpha=0.3)
    ax.set_xlim(0, 1)
fig.suptitle('Test signals', y=1.05)
plt.tight_layout()
plt.show()

## Fourier truncation and approximation error

We compute the bandlimited reconstruction $f_r$ for increasing $r$ and measure the approximation error $\|f - f_r\|_2$. On a log-log plot, the error slope reveals the convergence rate.

In [ ]:
def fourier_approx_1d(sig, r):
    F = np.fft.rfft(sig)
    F_trunc = F.copy()
    F_trunc[r+1:] = 0
    return np.fft.irfft(F_trunc, n=len(sig))


r_values = np.unique(np.round(np.logspace(0, np.log10(n//2), 60)).astype(int))
errors = {name: [] for name in signals}
for name, sig in signals.items():
    for r in r_values:
        approx = fourier_approx_1d(sig, r)
        errors[name].append(np.linalg.norm(sig - approx) / np.sqrt(n))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
cols = plt.cm.tab10(np.linspace(0, 0.4, 4))
for (name, err), col in zip(errors.items(), cols):
    axes[0].loglog(r_values, err, '-o', ms=3, lw=2, color=col, label=name)
axes[0].set_xlabel('bandwidth $r$'); axes[0].set_ylabel('$\\|f - f_r\\|_2$')
axes[0].set_title('Approximation error vs bandwidth')
axes[0].legend(fontsize=9); axes[0].grid(alpha=0.3, which='both')

# Visual comparison for one signal
r_show = [4, 16, 64, 256]
name_show = 'box'
sig_show = signals[name_show]
for i, r in enumerate(r_show):
    approx = fourier_approx_1d(sig_show, r)
    t_norm = i / (len(r_show) - 1)
    axes[1].plot(t, approx + 0, lw=1.8, color=(t_norm, 0, 1-t_norm),
                 label=f'$r={r}$', alpha=0.9)
axes[1].plot(t, sig_show, 'k--', lw=1, alpha=0.5, label='original')
axes[1].set_xlim(0, 1); axes[1].set_title(f'{name_show}: Gibbs oscillations near jump')
axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 2D image Fourier approximation

For a 2D image, we retain all Fourier coefficients within a disk of radius $r$ in frequency space and set all others to zero. The reconstruction shows how the image is built up from coarse structure to fine detail.

In [ ]:
# Build a synthetic test image
n2 = 128
x2 = np.linspace(0, 1, n2)
X2, Y2 = np.meshgrid(x2, x2)

# Smooth texture with edges
img = (0.5 * np.sin(8 * np.pi * X2) * np.cos(6 * np.pi * Y2)
       + 0.3 * (np.abs(X2 - 0.4) < 0.15) * (np.abs(Y2 - 0.6) < 0.15)
       + 0.2 * np.exp(-((X2-0.7)**2 + (Y2-0.3)**2) / 0.01))
img = (img - img.min()) / (img.max() - img.min())

# Frequency disk mask
kx2 = np.fft.fftfreq(n2) * n2
ky2 = np.fft.fftfreq(n2) * n2
KX, KY = np.meshgrid(kx2, ky2)
R_freq = np.sqrt(KX**2 + KY**2)

def fourier_approx_2d(img, r):
    F = np.fft.fft2(img)
    return np.fft.ifft2(F * (R_freq <= r)).real

r_show_2d = [2, 5, 10, 20, 40, n2//2]
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for ax, r in zip(axes.ravel(), r_show_2d):
    approx = fourier_approx_2d(img, r)
    ax.imshow(approx, cmap='gray', vmin=0, vmax=1)
    n_modes = int(np.sum(R_freq <= r))
    ax.set_title(f'$r={r}$  ({n_modes} modes)', fontsize=9)
    ax.axis('off')
fig.suptitle('2D Fourier approximation: disk of modes in frequency space', y=1.02)
plt.tight_layout()
plt.show()

## Interactive: 1D and 2D frequency sweep

Use the slider to increase the bandwidth $r$ and watch the 1D signal and 2D image reconstruction improve.

In [ ]:
from ipywidgets import Dropdown

def show_approx(signal_name='box', r_1d=20, r_2d=15):
    sig = signals[signal_name]
    approx_1d = fourier_approx_1d(sig, r_1d)
    approx_2d = fourier_approx_2d(img, r_2d)
    err_1d = np.linalg.norm(sig - approx_1d) / np.sqrt(n)
    frac_modes = np.sum(R_freq <= r_2d) / n2**2

    fig, axes = plt.subplots(1, 3, figsize=(13, 4))
    axes[0].plot(t, sig, 'gray', lw=1.5, alpha=0.7, label='original')
    axes[0].plot(t, approx_1d, 'b-', lw=2, label=f'$r={r_1d}$')
    axes[0].set_title(f'{signal_name}:  $\\|f-f_r\\|={err_1d:.4f}$')
    axes[0].legend(fontsize=9); axes[0].grid(alpha=0.3)

    axes[1].imshow(approx_2d, cmap='gray', vmin=0, vmax=1)
    axes[1].set_title(f'2D approx $r={r_2d}$  ({100*frac_modes:.1f}% of modes)')
    axes[1].axis('off')

    # Frequency domain mask
    mask = (R_freq <= r_2d).astype(float)
    axes[2].imshow(np.fft.fftshift(mask), cmap='Blues', origin='lower')
    axes[2].set_title('Retained frequencies (disk)')
    axes[2].axis('off')
    plt.tight_layout(); plt.show()

interact(
    show_approx,
    signal_name=Dropdown(options=list(signals.keys()), description='signal'),
    r_1d=IntSlider(value=20, min=1, max=n//4, step=5, description='$r$ (1D)'),
    r_2d=IntSlider(value=15, min=1, max=n2//2, step=2, description='$r$ (2D)'),
);

## Bibliographical resources

- Zygmund, A. (2002). *Trigonometric Series* (3rd ed.). Cambridge University Press.
- Körner, T. W. (1988). *Fourier Analysis*. Cambridge University Press.
- Mallat, S. (2009). *A Wavelet Tour of Signal Processing* (3rd ed.). Academic Press.
- Trefethen, L. N. (2000). *Spectral Methods in MATLAB*. SIAM.
- Katznelson, Y. (2004). *An Introduction to Harmonic Analysis* (3rd ed.). Cambridge University Press.